In [0]:
-- Unified Bright TV Dataset - All Metrics Combined
-- Date Range: 1 JAN 2016 - 31 MARCH 2016

WITH

  -- Step 1: Join user profiles with viewership data
  base_data AS (
    SELECT
      COALESCE(p.UserID, v.UserID0) AS UserID,
      p.Age,
      CASE
        WHEN p.Gender IS NULL OR TRIM(p.Gender) = '' OR p.Gender = 'None' THEN 'Unknown'
        ELSE p.Gender
      END AS Gender,
      p.Province,
      p.Race,
      v.RecordDate2,
      v.`Duration 2` AS Duration,
      v.Channel2 AS Channel
    FROM workspace.default.bright_tv_user_profiles AS p
    FULL OUTER JOIN workspace.default.bright_tv_viewership AS v
      ON p.UserID = v.UserID0
  ),
  
  -- Step 2: Add time conversions and components
  with_time AS (
    SELECT
      *,
      split(RecordDate2, ' ')[0] AS date,
      split(RecordDate2, ' ')[1] AS time,
      from_utc_timestamp(RecordDate2, 'Africa/Johannesburg') AS time_south_africa
    FROM base_data
  ),
  
  -- Step 3: Extract duration metrics
  with_duration AS (
    SELECT
      *,
      -- Extract duration components from timestamp
      hour(Duration) AS duration_hours,
      minute(Duration) AS duration_minutes,
      second(Duration) AS duration_seconds,
      
      -- Calculate total duration in seconds and minutes
      (hour(Duration) * 3600 + minute(Duration) * 60 + second(Duration)) AS duration_total_seconds,
      ROUND((hour(Duration) * 3600 + minute(Duration) * 60 + second(Duration)) / 60.0, 2) AS duration_total_minutes,
      
      -- Duration categories
      CASE
        WHEN Duration IS NULL THEN 'Unknown'
        WHEN (hour(Duration) * 3600 + minute(Duration) * 60 + second(Duration)) < 300 THEN 'short (< 5 min)'
        WHEN (hour(Duration) * 3600 + minute(Duration) * 60 + second(Duration)) >= 300 
         AND (hour(Duration) * 3600 + minute(Duration) * 60 + second(Duration)) < 1800 THEN 'medium (5-30 min)'
        WHEN (hour(Duration) * 3600 + minute(Duration) * 60 + second(Duration)) >= 1800 THEN 'long (30+ min)'
        ELSE 'Unknown'
      END AS duration_category
    FROM with_time
  ),
  
  -- Step 4: Add categorical fields
  final_data AS (
    SELECT
      *,
      -- Age categories
      CASE
        WHEN Age IS NULL OR Age = 0 THEN 'Unknown'
        WHEN Age <= 18 THEN 'minors'
        WHEN Age > 18 AND Age <= 30 THEN 'young adults'
        WHEN Age > 30 AND Age <= 45 THEN 'adults'
        WHEN Age > 45 AND Age <= 60 THEN 'middle aged'
        WHEN Age > 60 THEN 'seniors'
        ELSE 'Unknown'
      END AS age_category,
      
      -- Gender categories
      CASE
        WHEN Gender = 'Unknown' THEN 'Unknown'
        WHEN LOWER(Gender) = 'male' THEN 'male'
        WHEN LOWER(Gender) = 'female' THEN 'female'
        ELSE 'Unknown'
      END AS gender_category,
      
      -- Time buckets (South Africa time)
      CASE
        WHEN time_south_africa IS NULL THEN 'Unknown'
        WHEN hour(time_south_africa) BETWEEN 0 AND 5 THEN 'early morning'
        WHEN hour(time_south_africa) BETWEEN 6 AND 11 THEN 'morning'
        WHEN hour(time_south_africa) BETWEEN 12 AND 17 THEN 'afternoon'
        WHEN hour(time_south_africa) BETWEEN 18 AND 23 THEN 'evening'
        ELSE 'Unknown'
      END AS time_buckets,
      
      -- Month name
      CASE
        WHEN time_south_africa IS NULL THEN 'Unknown'
        ELSE date_format(time_south_africa, 'MMMM')
      END AS month_name,
      
      -- Day name
      CASE
        WHEN time_south_africa IS NULL THEN 'Unknown'
        ELSE date_format(time_south_africa, 'EEEE')
      END AS day_name,
      
      -- Weekend/Weekday indicator
      CASE
        WHEN time_south_africa IS NULL THEN 'Unknown'
        WHEN dayofweek(time_south_africa) IN (1, 7) THEN 'weekend'
        ELSE 'weekday'
      END AS day_type
    FROM with_duration
  ),
  
  -- Aggregate: Channel Sessions Count
  channel_stats AS (
    SELECT 
      Channel,
      COUNT(*) AS channel_sessions_count
    FROM final_data
    WHERE Channel IS NOT NULL
    GROUP BY Channel
  ),
  
  -- Aggregate: Hourly Sessions Count
  hourly_stats AS (
    SELECT 
      hour(time_south_africa) AS hour_sa,
      COUNT(*) AS hourly_sessions_count
    FROM final_data
    WHERE time_south_africa IS NOT NULL
    GROUP BY hour(time_south_africa)
  ),
  
  -- Aggregate: Daily Sessions Count
  daily_stats AS (
    SELECT 
      date,
      COUNT(*) AS daily_sessions_count
    FROM final_data
    WHERE date IS NOT NULL
    GROUP BY date
  ),
  
  -- Global metrics
  global_metrics AS (
    SELECT 
      COUNT(*) AS total_sessions,
      COUNT(DISTINCT UserID) AS total_subscribers
    FROM final_data
    WHERE UserID IS NOT NULL
      AND RecordDate2 IS NOT NULL
      AND Duration IS NOT NULL
      AND Channel IS NOT NULL
  )

-- Final unified result with all metrics joined
SELECT 
  fd.*,
  -- Add channel metrics
  cs.channel_sessions_count,
  -- Add hourly metrics
  hs.hourly_sessions_count,
  -- Add daily metrics
  ds.daily_sessions_count,
  -- Add global metrics
  gm.total_sessions,
  gm.total_subscribers
FROM final_data fd
LEFT JOIN channel_stats cs ON fd.Channel = cs.Channel
LEFT JOIN hourly_stats hs ON hour(fd.time_south_africa) = hs.hour_sa
LEFT JOIN daily_stats ds ON fd.date = ds.date
CROSS JOIN global_metrics gm
WHERE fd.UserID IS NOT NULL
  AND fd.RecordDate2 IS NOT NULL
  AND fd.Duration IS NOT NULL
  AND fd.Channel IS NOT NULL;
